In [95]:
import math
import matplotlib.pyplot as plt
from numpy import arange, asarray, exp, array, float32
from numpy.random import normal
import numpy as np

from PySide6.QtWidgets import QApplication, QWidget, QPushButton, QMainWindow, QGridLayout, QFrame
from PySide6.QtCore import Slot, QSize, Signal, QObject
import pyqtgraph as pg
from pyqtgraph import PlotWidget

from pipython import GCSDevice, pitools
from pipython.pidevice.gcsmessages import GCSMessages
from pipython.pidevice.interfaces.piserial import PISerial
from pipython.pidevice.gcscommands import GCSCommands
from sys import platform
import time

from matplotlib.backend_bases import key_press_handler
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure

In [85]:
def gaussian(x, sigma, mu):
    x = asarray(x)
    #y= (1/(sigma * math.sqrt(2 * math.pi )) * math.exp(-1/2 * (x-mu)**2 / sigma**2))
    return asarray(1/(sigma * math.sqrt(2 * math.pi )) * exp(-1/2 * (x-mu)**2 / sigma**2))

def define_x_grid(min, max, resolution):

    x = arange(min, max+resolution, resolution)

    return x

def create_random_gaussian(with_noise = False):
    random_x = np.random.Generator.random(-10, 10, size=None)
    y = gaussian(random_x, 1, 0)
    if with_noise == True:
        y += float(normal(scale=0.1, size=None))
    return random_x, y

def append_point(x, y, x_new, y_new):
    x.append(x_new)
    y.append(y_new)
    

In [113]:
class Spectral_data(QObject):
    data_changed = Signal()

    def __init__(self):
        super().__init__()
        self.data = np.empty((2,0), dtype=float32)
        self.rng = np.random.default_rng()

    def add_point(self, x, y):
        point = asarray([[x],[y]], dtype=float32)
        self.data = np.append(self.data, point, axis=1)
        self.data_changed.emit()

    def print_data(self):
        print(self.data)

    def clear_data(self):
        self.data = np.empty((2,0), dtype=float32)
        self.data_changed.emit()

    def add_random_point(self):
        new_x = self.rng.uniform(low=-5, high=5)
        new_y = gaussian(new_x, 1.44, 0)
        self.add_point(new_x, new_y)
        self.data_changed.emit()

    def add_noise(self):
        self.data[1,:] += self.rng.normal(scale=0.01, size=len(self.data[1,:]))
        self.data_changed.emit()


class Spectral_plot(PlotWidget):
    def __init__(self):
        super().__init__()
        self.plot_points = self.plot([],[], pen=None, symbol="o")# = Figure(dpi=75)
     #   self.ax = self.fig.add_subplot()
     #   self.plot_points = self.ax.scatter([], [])
     #   self.window = window
     #   self.window.columnconfigure(0, weight=1)
     #   self.window.rowconfigure(0, weight=1)

     #   self.canvas = FigureCanvasTkAgg(self.fig, master=self.window)
     #   self.canvas.draw()
     #   self.canvas.get_tk_widget().grid(row=0, column=0, sticky="nsew")       

    #def plot_spectrum(self):
    #    plt.scatter(spectrum.data[0,:], spectrum.data[1,:])

    #@Slot()
    def update_plot(self, data_x, data_y):
        self.plot_points.setData(data_x, data_y)
  #      self.ax.update_datalim(data.T, updatex=True, updatey=True)
  #      self.ax.autoscale_view()
  #      self.fig.canvas.draw_idle()

    #def clear_plot(self):
    #    self.update_plot(np.empty((2,0), dtype=float32))

class Control_stage():
    def __init__(self):
       # if platform == "linux" or platform == "linux2":
       #     self.port = '/dev/ttyS0',
       # elif platform == "win32":
       #     self.port='COM1'

        self.pidevice = GCSDevice()
        self.device = None
        self.devices_list = ["test"]

    def get_devices(self):
        self.devices_list = list(self.pidevice.EnumerateTCPIPDevices(mask='C-884.4DB'))
        if len(self.devices_list) != 0:
            return self.devices_list
        else:
            return []
    
    def connect_device(self, device):
        self.pidevice.ConnectTCPIPByDescription(device)

    def print_identity(self):
        self.pidevice.qIDN()

    def move_stage_to_z(self, z_position):
        pitools.moveandwait(self.pidevice, 'Axis_1', float32(z_position))

class Control_stage_fake():
    def __init__(self):
        self.pidevice = "test"
        self.pos = 0
        self.device = None
        self.devices_list = ["test"]

    def get_devices(self):
        self.devices_list = ["Device 1", "Device 2", "Device 3", "Device 4"]
        if len(self.devices_list) != 0:
            return self.devices_list
        else:
            return []
    
    def connect_device(self, device):
        self.pidevice = device

    def print_identity(self):
        return self.pidevice

    def move_stage_to_z(self, z_position):
        self.pos = z_position
        time.sleep(1)

class Plotcontrolpanel(QFrame):
    ################# Signals #################
    request_add_point = Signal()
    request_shutdown  = Signal()
    request_clear     = Signal()
    request_noise     = Signal()

    def __init__(self, parent):
        super().__init__(parent)
        plot_control_layout = QGridLayout()
        self.setLayout(plot_control_layout)




        ################# Buttons #################

        self.off_button = QPushButton("Off")
        self.off_button.clicked.connect(self.request_shutdown.emit)
        plot_control_layout.addWidget(self.off_button, 0, 0)

        self.add_point_button =QPushButton("add")
        self.add_point_button.clicked.connect(self.request_add_point.emit)
        plot_control_layout.addWidget(self.add_point_button, 1, 0)



        self.clear_button = QPushButton("Clear")
        self.clear_button.clicked.connect(self.request_clear.emit)
        plot_control_layout.addWidget(self.clear_button, 2, 0)

        self.noise_button = QPushButton("Add Noise")
        self.noise_button.clicked.connect(self.request_noise.emit)
        plot_control_layout.addWidget(self.noise_button, 3, 0)


class Stagecontrolpanel(QFrame):
    def __init__(self, parent):
        super().__init__(parent)
        stage_control_layout = QGridLayout()
        self.setLayout(stage_control_layout)


class Main_window(QMainWindow):
    def __init__(self, data = Spectral_data(), stage = Control_stage_fake()):
        super().__init__()

        self.setWindowTitle("Data Aquisition")
        self.setMinimumSize(QSize(400,300))

        self.data = data
        self.stage = stage 

        central_widget = QWidget()
        self.setCentralWidget(central_widget)

        central_layout = QGridLayout()
        central_widget.setLayout(central_layout)

        self.plot_widget = Spectral_plot()
        central_layout.addWidget(self.plot_widget, 0, 1)

        self.plot_controlpanel = Plotcontrolpanel(self)
        central_layout.addWidget(self.plot_controlpanel, 0, 0)

        self.plot_controlpanel.request_add_point.connect(self.add_random_point)
        self.plot_controlpanel.request_shutdown.connect(self.shutdown)
        self.plot_controlpanel.request_clear.connect(self.clear_data)
        self.plot_controlpanel.request_noise.connect(self.add_noise)


########################################################## Frame 1 ##########################################################
#
#        self.noise_button = ttk.Button(self.frame1, text="noise", command=self.add_noise)
#        self.noise_button.grid(row      = 1, 
#                               column   = 1,
#                               padx     = 5, 
#                               pady     = 5, 
#                               sticky   = "ew")
#
########################################################## Frame 2 ##########################################################
#        self.frame2 = ttk.Frame(self.frame_controls)
#        self.frame2.grid(row    = 1, 
#                         column = 0,
#                         padx   = 5, 
#                         pady   = 5, 
#                         sticky = "nsew")
#        
#
#        self.z_stage_label = ttk.Label(self.frame2, text="Position Stage:")
#        self.z_stage_label.grid(row     = 0, 
#                                column  = 0,
#                                padx    = 5, 
#                                pady    = 5)
#
#        self.enter_z_stage = ttk.Entry(self.frame2)
#        self.enter_z_stage.grid(row     = 0, 
#                                column  = 1,
#                                padx    = 5, 
#                                pady    = 5, 
#                                sticky  = "ew")
#        
#        self.get_devices = ttk.Button(self.frame2, text="Get Devices", command=self.update_device_list)
#        self.get_devices.grid(row      = 1, 
#                               column   = 0,
#                               padx     = 5, 
#                               pady     = 5, 
#                               sticky   = "ew")
#
#
#        self.z_stage_select_controler = ttk.Combobox(self.frame2, state="readonly", textvariable=self.stage.device, values=self.stage.devices_list)
#        self.z_stage_select_controler.grid( row     = 1, 
#                                            column  = 1,
#                                            padx    = 5, 
#                                            pady    = 5, 
#                                            sticky  = "ew")

#    def update_device_list(self):
#        self.stage.get_devices()
#        self.z_stage_select_controler["values"] = self.stage.devices_list
    @Slot()
    def add_random_point(self):
        data.add_random_point()
        self.plot_widget.update_plot(self.data.data[0], self.data.data[1])

    @Slot()
    def add_noise(self):
        data.add_noise()
        self.plot_widget.update_plot(self.data.data[0], self.data.data[1])

    @Slot()
    def clear_data(self):
        data.clear_data()
        self.plot_widget.update_plot([], [])

    @Slot()
    def shutdown(self):
        self.close()



In [114]:
stage = Control_stage_fake()
#app.shutdown()
%gui qt
#gui.app.shutdown()



app = QApplication.instance()
#window = QWidget()
data = Spectral_data()


window = Main_window(data=data, stage= stage)

window.show()
#gui.app.exec()

